# Evaluation Module — src/evaluate.py

**Purpose**: Demonstrate the evaluation module on the HOG+SVM baseline results and explain the metric choices.

**Why a dedicated evaluation module?**  
Both pipelines (HOG+SVM and EfficientNet) will use identical metrics so results are directly comparable. Centralizing evaluation prevents inconsistencies — for example, accidentally using `weighted` F1 for one model and `binary` F1 for the other.

---

### Why not just use accuracy?

The MVTec `metal_nut` test set has ~73 good and ~28 defective images (72%/28% split).  
A classifier that **always predicts 'good'** achieves **72% accuracy** — without detecting a single defect.  
This is why we report **recall on the defective class** as the primary safety metric:
it directly measures what fraction of defects the model actually catches.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

from src.preprocessing import preprocess
from src.features import extract_hog
from src.models.classical import ClassicalClassifier
from src.evaluate import evaluate_classification, print_results, results_row

print('Imports OK')

## 1. Reproduce HOG+SVM results using evaluate.py

In [ ]:
CATEGORY   = 'metal_nut'
DATA_ROOT  = Path('../data/mvtec_ad')
RANDOM_STATE = 42

def load_split(folder, label):
    images, labels = [], []
    for p in sorted(folder.glob('*.png')):
        images.append(extract_hog(preprocess(str(p))))
        labels.append(label)
    return images, labels

category_path = DATA_ROOT / CATEGORY
all_X, all_y = [], []

for folder in [category_path / 'train' / 'good', category_path / 'test' / 'good']:
    X, y = load_split(folder, 0)
    all_X += X; all_y += y

for subfolder in sorted((category_path / 'test').iterdir()):
    if subfolder.is_dir() and subfolder.name != 'good':
        X, y = load_split(subfolder, 1)
        all_X += X; all_y += y

all_X = np.array(all_X)
all_y = np.array(all_y)

X_train, X_test, y_train, y_test = train_test_split(
    all_X, all_y, test_size=0.30, random_state=RANDOM_STATE, stratify=all_y
)

clf = ClassicalClassifier().fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('Data loaded and model trained.')

## 2. Evaluate with the standardized module

In [ ]:
metrics = evaluate_classification(y_test, y_pred, model_name='HOG + SVM (metal_nut)')
print_results(metrics)

## 3. Results row — format for the comparison table

This is the format used in `notebooks/05_model_comparison.ipynb` to build the final side-by-side table.

In [ ]:
import pandas as pd

row = results_row(metrics)
df  = pd.DataFrame([row])
print(df.to_string(index=False))

## 4. Metric decisions — summary

| Metric | Formula | Why we report it |
|---|---|---|
| Accuracy | (TP+TN) / N | Overall correctness — misleading with imbalanced classes |
| F1 (binary) | 2·P·R / (P+R) on defect class | Balances precision and recall for defects specifically |
| Recall (defect) | TP / (TP+FN) | Fraction of defects caught — primary safety metric |
| Precision (defect) | TP / (TP+FP) | Fraction of defect alerts that are real |
| F1 (weighted) | class-weighted average | Overall model quality accounting for imbalance |
| IoU / Dice | mask overlap | For future segmentation extension only |

**The industrial priority order**: Recall (defect) > F1 (binary) > Accuracy